## Synthetic retailer customer analytics

Thornbury & Co. is fictional. Seed 42 generates 2,500 customers, 9,632 invoices and 29,336 line items. Budget, costs and responses are assumptions; no actual marketing returns or full channel-rank inversion are evidenced.


In [ ]:
import sys, yaml, json
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.clv.etl import (
    load_synthetic, load_uci_online_retail,
    split_calibration_holdout, to_summary, get_customer_channels,
)
from src.clv.rfm import compute_rfm, umap_cluster, segment_summary, rfm_channel_profile
from src.clv.bgf import (
    fit_bgnbd, fit_gamma_gamma, predict_clv, clv_summary_stats,
    clv_by_channel, holdout_comparison, campaign_heuristic_by_channel,
)
from src.clv.cohorts import (
    build_retention_matrix, cohort_revenue_matrix,
    retention_by_channel, month1_retention_summary,
)
from src.clv.visualise import (
    plot_retention_heatmap, plot_clv_distribution,
    plot_segment_bar, plot_umap_scatter,
    plot_channel_clv_boxplot, plot_rfm_vs_clv_tension,
    plot_month1_retention_by_channel, plot_holdout_validation,
    plot_campaign_roi, plot_bgnbd_matrices,
)

OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

cfg = yaml.safe_load((ROOT / 'config' / 'default.yaml').read_text())
sns.set_theme(style='whitegrid')
print('Config loaded.')
print(f"  Scenario: {cfg['scenario']['name']}")
print(f"  Channels: {cfg['scenario']['channels']}")
print(f"  Win-back budget: £{cfg['campaign']['win_back_budget']:,}")

## Generated transactions

The optional UCI file was absent in the saved execution. Generated channel labels and shared engagement create purchase, spend and dropout dependence. Those assumptions are not empirical validation.


In [2]:
UCI_PATH = ROOT / 'data' / 'online_retail_II.xlsx'

if UCI_PATH.exists():
    print('Loading UCI Online Retail II ...')
    df = load_uci_online_retail(UCI_PATH)
    print('Note: UCI data has no channel labels, disaggregated analysis requires synthetic data.')
else:
    print('UCI file not found, using Thornbury & Co. synthetic data.')
    df = load_synthetic(
        n_customers=cfg['data']['synthetic_n_customers'],
        seed=cfg['data']['seed'],
    )

OBS_END = pd.Timestamp(cfg['data']['observation_end'])
CAL_END = OBS_END - pd.DateOffset(months=cfg['data']['holdout_months'])
SNAPSHOT = OBS_END

print(f'\nTransactions : {len(df):,}')
print(f'Customers    : {df["Customer ID"].nunique():,}')
print(f'Observation  : {df["InvoiceDate"].min().date()} → {OBS_END.date()}')
print(f'Calibration  : up to {CAL_END.date()}')
print(f'Holdout      : {CAL_END.date()} → {OBS_END.date()}')
df.head(3)

UCI file not found, using Thornbury & Co. synthetic data.

Transactions : 29,336
Customers    : 2,500
Observation  : 2010-01-01 → 2011-12-09
Calibration  : up to 2011-09-09
Holdout      : 2011-09-09 → 2011-12-09


,Customer ID,Channel,Invoice,InvoiceDate,Quantity,Price,Revenue,Country
0,C00001,Deal-Seeker,INV0000001,2010-09-19,1,46.36,46.36,Germany
1,C00001,Deal-Seeker,INV0000001,2010-09-19,1,48.19,48.19,Germany
2,C00001,Deal-Seeker,INV0000001,2010-09-19,1,72.74,72.74,Germany


## Customer summaries

Line items and invoices are different units. Generated summaries describe the fictional scenario.


In [3]:
# Channel composition
channel_map = get_customer_channels(df)
has_channels = not channel_map.empty

if has_channels:
    ch_counts = channel_map.value_counts()
    print('Customer counts by acquisition channel:')
    for ch, n in ch_counts.items():
        print(f'  {ch:<20} {n:>5d} customers ({n/len(channel_map):.1%})')

# Per-customer invoice frequency
cust_stats = (
    df.groupby('Customer ID')
    .agg(
        n_invoices=('Invoice', 'nunique'),
        total_revenue=('Revenue', 'sum'),
        avg_basket=('Revenue', lambda x: df.loc[x.index, 'Revenue'].sum() / df.loc[x.index, 'Invoice'].nunique()),
    )
)
if has_channels:
    cust_stats['Channel'] = cust_stats.index.map(channel_map)

print(f'\nOverall per-customer stats:')
print(f'  Median invoices    : {cust_stats["n_invoices"].median():.1f}')
print(f'  Median total rev   : £{cust_stats["total_revenue"].median():.2f}')

if has_channels:
    print('\nMean invoices per customer by channel:')
    print(cust_stats.groupby('Channel')['n_invoices'].mean().round(2).to_string())

Customer counts by acquisition channel:
  Mainstream            1097 customers (43.9%)
  Enthusiast             757 customers (30.3%)
  Deal-Seeker            646 customers (25.8%)

Overall per-customer stats:
  Median invoices    : 2.0
  Median total rev   : £302.37

Mean invoices per customer by channel:
Channel
Deal-Seeker    1.40
Enthusiast     7.13
Mainstream     3.04


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Transaction volume by month
df['InvoiceDate'].dt.to_period('M').value_counts().sort_index().plot(
    ax=axes[0], kind='bar', color='#4472C4'
)
axes[0].set_title('Monthly Transaction Volume')
axes[0].set_xlabel('Month')
axes[0].tick_params(axis='x', rotation=45)

# Orders per customer by channel
if has_channels:
    channel_order = ['Enthusiast', 'Mainstream', 'Deal-Seeker']
    colours = {'Enthusiast': '#2166ac', 'Mainstream': '#4dac26', 'Deal-Seeker': '#d7191c'}
    for ch in channel_order:
        if ch in cust_stats['Channel'].values:
            sub = cust_stats[cust_stats['Channel'] == ch]['n_invoices'].clip(upper=20)
            axes[1].hist(sub, bins=20, alpha=0.65, label=ch, color=colours[ch], edgecolor='white')
    axes[1].legend()
else:
    cust_stats['n_invoices'].clip(upper=20).hist(bins=20, ax=axes[1], color='#ED7D31', edgecolor='white')
axes[1].set_title('Invoices per Customer by Channel (cap 20)')
axes[1].set_xlabel('Number of Invoices')

# Revenue per customer distribution (log scale)
axes[2].hist(
    np.log1p(cust_stats['total_revenue'].clip(upper=cust_stats['total_revenue'].quantile(0.95))),
    bins=40, color='#70AD47', edgecolor='white'
)
axes[2].set_title('Log(Revenue) per Customer (95th pct cap)')
axes[2].set_xlabel('log(1 + Total Revenue)')

plt.tight_layout()
fig.savefig(OUTPUTS / 'eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('Revenue distribution is right-skewed, log-normal, as expected for retail spend.')

Revenue distribution is right-skewed, log-normal, as expected for retail spend.


## Cohort summaries

Month-1 values average available cohort cells. Observed zero-purchase months are currently omitted, unlike genuinely unobserved periods. Do not interpret these denominators as acquisition efficiency or business guardrails.


In [5]:
retention = build_retention_matrix(df, freq=cfg['cohorts']['frequency'])

min_size = cfg['cohorts']['min_cohort_size']
cohort_sizes = df.groupby(
    df['InvoiceDate'].dt.to_period(cfg['cohorts']['frequency'])
)['Customer ID'].nunique()
large_cohorts = cohort_sizes[cohort_sizes >= min_size].index
retention_display = retention.loc[retention.index.isin(large_cohorts)].iloc[:, :12]

fig = plot_retention_heatmap(
    retention_display,
    title='Thornbury & Co., Monthly Cohort Retention',
    save_path=OUTPUTS / 'retention_heatmap.png',
)
plt.show()

month1_overall = retention_display[1].dropna().mean() if 1 in retention_display.columns else float('nan')
print(f'Overall mean Month-1 retention: {month1_overall:.1%}')
guardrail = cfg['guardrail']['min_month1_retention']
print(f'Guardrail threshold           : {guardrail:.0%}')
print(f'Status                        : {"✓ PASS" if month1_overall >= guardrail else "✗ BREACH"}')

Overall mean Month-1 retention: 16.1%
Guardrail threshold           : 20%
Status                        : ✗ BREACH


## Scenario retention threshold

The 20% floor is an assumed scenario setting. The current mean excludes observed zero cells and does not demonstrate an actual commercial guardrail breach.


In [ ]:
if has_channels:
    ret_summary = month1_retention_summary(df, freq=cfg['cohorts']['frequency'])
    print('Month-1 retention by acquisition channel:')
    print(ret_summary.to_string(index=False))

    fig = plot_month1_retention_by_channel(
        ret_summary,
        guardrail=guardrail,
        save_path=OUTPUTS / 'month1_retention_by_channel.png',
    )
    plt.show()

    deal_ret = ret_summary.loc[ret_summary['Channel'] == 'Deal-Seeker', 'mean_month1_retention']
    if len(deal_ret):
        dv = float(deal_ret.iloc[0])
        print(f'\nDeal-Seeker month-1 retention: {dv:.1%}')
        if dv < guardrail:
            print(f'  → GUARDRAIL BREACHED ({dv:.1%} < {guardrail:.0%})')
            print('  This synthetic comparison does not establish acquisition efficiency or observed loyalty.')
else:
    print('Channel data not available, guardrail check requires synthetic or augmented data.')

## RFM and chronological calibration

RFM describes recency, invoice frequency and historical Monetary. Mean M ranks Enthusiasts highest at 3.55, then Deal-Seekers 2.79 and Mainstream 2.74. Models fit through 9 September 2011. The following 91 days through 9 December form a purchase-count holdout. The configured calibration_months field is unused. Count validation is separate from final-date future-value prediction.


In [7]:
rfm = compute_rfm(df, snapshot_date=SNAPSHOT, bins=cfg['rfm']['recency_bins'])
rfm = umap_cluster(rfm, n_clusters=cfg['rfm']['n_clusters'])
seg = segment_summary(rfm)

print('Segment summary (by total revenue):')
print(seg.to_string(index=False))

d:\Users\jesco\Documents\Github\customer-lifetime-value\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Users\jesco\Documents\Github\customer-lifetime-value\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Segment summary (by total revenue):
            Segment  Customers  Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue
    Loyal Customers        444        89.79           6.12        576.57      255997.73
          Champions        270        27.13          10.74        815.60      220213.21
Potential Loyalists        398       176.87           3.98        422.50      168156.50
               Lost        477       590.73           1.27        299.03      142636.46
        Hibernating        352       423.26           1.19        292.13      102830.22
            At Risk        146       420.53           3.43        363.36       53050.39
    Needs Attention        153       266.02           2.48        336.00       51408.75
     About to Sleep        145       297.07           1.74        324.50       47051.83
          Promising         55       121.40           2.00        403.40       22186.97
   Recent Customers         38        33.26           2.00        359.98       13679

In [ ]:
fig = plot_segment_bar(seg, save_path=OUTPUTS / 'segment_revenue.png')
plt.show()

fig2 = plot_umap_scatter(rfm, save_path=OUTPUTS / 'umap_scatter.png')
if fig2:
    plt.show()

seg.to_csv(OUTPUTS / 'segment_profiles.csv', index=False)

if has_channels:
    ch_profile = rfm_channel_profile(rfm)
    print('\nChannel profile (mean RFM scores):')
    print(ch_profile[['Channel', 'n_customers', 'mean_R', 'mean_F', 'mean_M',
                       'mean_Monetary', 'dominant_segment']].to_string(index=False))
    print('\nSaved mean M ranks Enthusiasts highest; inspect each summary without assuming an inversion.')
    print('This is the seed of the tension that the BG/NBD model will resolve.')

## BG/NBD purchase model

Fitted parameters on synthetic observations are not proof of independence assumptions. Alpha is a Gamma rate parameter; a and b are Beta shape parameters. Shared engagement links purchase rate and dropout.


In [9]:
cal_df, hold_df = split_calibration_holdout(df, CAL_END)

cal_summary = to_summary(cal_df, CAL_END)
print(f'Calibration customers : {len(cal_summary):,}')
print(f'Repeat buyers (freq>0): {(cal_summary["frequency"] > 0).sum():,} ({(cal_summary["frequency"] > 0).mean():.1%})')
print(f'Median frequency      : {cal_summary["frequency"].median():.1f}')
print(f'Median T (days)       : {cal_summary["T"].median():.0f}')
cal_summary.head(3)

Calibration customers : 2,500
Repeat buyers (freq>0): 1,714 (68.6%)
Median frequency      : 1.0
Median T (days)       : 432


,frequency,recency,T,monetary_value
Customer ID,,,,
C00001,0.0,0.0,355.0,0.0000
C00002,4.0,494.0,515.0,149.7975
C00003,0.0,0.0,315.0,0.0000


In [10]:
bgf = fit_bgnbd(cal_summary, penalizer_coef=cfg['clv']['penalizer_coef'])
print(bgf)
print(f'\nModel parameters:')
print(f'  r = {bgf.params_["r"]:.4f}  (purchase rate heterogeneity)')
print(f'  α = {bgf.params_["alpha"]:.4f}  (purchase rate scale)')
print(f'  a = {bgf.params_["a"]:.4f}  (dropout heterogeneity)')
print(f'  b = {bgf.params_["b"]:.4f}  (dropout scale)')
print(f'  Mean implied dropout prob: {bgf.params_["a"] / (bgf.params_["a"] + bgf.params_["b"]):.3f}')

fig = plot_bgnbd_matrices(bgf, save_path=OUTPUTS / 'bgnbd_matrices.png')
plt.show()

print('\nKey insight: a customer with high frequency but low recency (bottom-right) has LOW P(alive).')
print('This is exactly where Deal-Seekers cluster, they bought several times but not recently.')

<lifetimes.BetaGeoFitter: fitted with 2500 subjects, a: 0.21, alpha: 98.63, b: 1.38, r: 0.71>

Model parameters:
  r = 0.7083  (purchase rate heterogeneity)
  α = 98.6269  (purchase rate scale)
  a = 0.2118  (dropout heterogeneity)
  b = 1.3753  (dropout scale)
  Mean implied dropout prob: 0.133



Key insight: a customer with high frequency but low recency (bottom-right) has LOW P(alive).
This is exactly where Deal-Seekers cluster, they bought several times but not recently.


## Gamma-Gamma spend model

Fit uses 1,714 calibration-period repeat buyers. Frequency/value independence needs calibration-period diagnostics because the generator relates them. Final repeat-buyer correlation is descriptive, not a calibration diagnostic.


In [11]:
ggf = fit_gamma_gamma(cal_summary, penalizer_coef=cfg['clv']['penalizer_coef'])
print(ggf)

repeat = cal_summary[cal_summary['frequency'] > 0]
predicted_avg = ggf.conditional_expected_average_profit(repeat['frequency'], repeat['monetary_value'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cap = repeat['monetary_value'].quantile(0.98)
axes[0].scatter(repeat['monetary_value'].clip(upper=cap), predicted_avg.clip(upper=cap),
                alpha=0.2, s=6, color='#4472C4')
axes[0].plot([0, cap], [0, cap], 'r--', linewidth=1.5, label='Perfect calibration')
axes[0].set_xlabel('Observed Avg Revenue per Transaction (£)')
axes[0].set_ylabel('GGF Predicted Avg Revenue (£)')
axes[0].set_title('Gamma-Gamma: Calibration Check')
axes[0].legend()

# Spend distribution by channel
if has_channels:
    repeat2 = repeat.copy()
    repeat2['Channel'] = repeat2.index.map(channel_map)
    colours_ch = {'Enthusiast': '#2166ac', 'Mainstream': '#4dac26', 'Deal-Seeker': '#d7191c'}
    for ch in ['Enthusiast', 'Mainstream', 'Deal-Seeker']:
        sub = repeat2[repeat2['Channel'] == ch]['monetary_value'].clip(upper=cap)
        if len(sub):
            axes[1].hist(sub, bins=30, alpha=0.6, label=ch, color=colours_ch.get(ch, '#888'), edgecolor='white')
    axes[1].set_title('Avg Transaction Value by Channel (repeat buyers)')
    axes[1].set_xlabel('Avg Revenue per Transaction (£)')
    axes[1].legend()
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
else:
    axes[1].hist(repeat['monetary_value'].clip(upper=cap), bins=40, color='#ED7D31', edgecolor='white')
    axes[1].set_title('Avg Transaction Value Distribution')

plt.tight_layout()
fig.savefig(OUTPUTS / 'ggf_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'GGF fitted on {len(repeat):,} repeat buyers')
print(f'Customers excluded (single purchase): {(cal_summary["frequency"] == 0).sum():,}')

<lifetimes.GammaGammaFitter: fitted with 1714 subjects, p: 7.83, q: 1.24, v: 8.05>


GGF fitted on 1,714 repeat buyers
Customers excluded (single purchase): 786


## Discounted revenue estimates

Mean GBP143, median GBP63.25 and total GBP357,493.42 are nominal annual discounted revenues, not profit. The wrapper passes 365/30 monthly steps and the inspected helper can reach day 390; executed version is unrecorded. Nonrepeaters are forced to zero by policy. Final value has no monetary holdout.


In [ ]:
full_summary = to_summary(df, OBS_END)

clv_df = predict_clv(
    bgf, ggf, full_summary,
    horizon_days=cfg['clv']['horizon_days'],
    discount_rate=cfg['clv']['discount_rate'],
)

stats = clv_summary_stats(clv_df)
print('Nominal annual discounted revenue summary (endpoint unverified):')
for k, v in stats.items():
    label = k.replace('_', ' ').title()
    if 'clv' in k or 'revenue' in k:
        print(f'  {label:<35} £{v:,.2f}')
    else:
        print(f'  {label:<35} {v}')

top20 = (
    clv_df.sort_values('predicted_clv', ascending=False)
    .head(20)[['frequency', 'recency', 'T', 'monetary_value', 'predicted_purchases', 'prob_alive', 'predicted_clv']]
)
print(f'\nTop 5 customers by nominal annual discounted revenue:')
top20.head(5)

In [ ]:
fig = plot_clv_distribution(clv_df, save_path=OUTPUTS / 'clv_distribution.png')
plt.show()

clv_df.to_csv(OUTPUTS / 'clv_predictions.csv')

# Pareto check: what % of customers generate 80% of predicted revenue?
sorted_clv = clv_df['predicted_clv'].sort_values(ascending=False)
cumulative = sorted_clv.cumsum() / sorted_clv.sum()
pct_for_80 = (int(np.searchsorted(cumulative.to_numpy(), 0.80)) + 1) / len(sorted_clv)
print(f'Pareto: top {pct_for_80:.1%} of customers generate 80% of nominal annual discounted revenue.')
print(f'Mean CLV : £{stats["mean_clv"]:,.2f}')
print(f'Median CLV: £{stats["median_clv"]:,.2f}  (mean >> median → heavy right tail)')

## RFM and predicted value

Saved RFM Monetary and CLV both rank Enthusiasts highest. A smaller Deal-Seeker/Mainstream reversal does not establish a full inversion or a budget rule.


In [ ]:
if has_channels:
    fig = plot_rfm_vs_clv_tension(
        rfm_df=rfm,
        clv_df=clv_df,
        channel_map=channel_map,
        save_path=OUTPUTS / 'rfm_vs_clv_tension.png',
    )
    plt.show()

    # Quantify the conflict
    clv_ch = clv_df.copy()
    clv_ch['Channel'] = clv_ch.index.map(channel_map)
    rfm_ch = rfm.set_index('Customer ID') if 'Customer ID' in rfm.columns else rfm
    rfm_ch = rfm_ch[['M', 'Monetary']].join(clv_ch[['predicted_clv', 'prob_alive', 'Channel']], how='inner')

    print('Mean M score vs mean predicted CLV by channel:')
    conflict = rfm_ch.groupby('Channel').agg(
        mean_M=('M', 'mean'),
        mean_monetary=('Monetary', 'mean'),
        mean_clv=('predicted_clv', 'mean'),
        mean_prob_alive=('prob_alive', 'mean'),
    ).round(2)
    print(conflict.to_string())
    print('\nNote: The saved run ranks Enthusiasts highest on both mean Monetary and predicted CLV.')
    print('There is no full rank inversion or validated campaign budget rule.')
else:
    print('Channel data not available, tension plot requires synthetic or augmented data.')

## Purchase-count holdout

Full-population MAE=.4745 and signed bias=+.0348 validate only purchase counts. Plotted diagnostics trim above the 98th percentile of actual counts. Aggregate bias can cancel; channel MAE is not channel bias.


In [ ]:
comparison = holdout_comparison(bgf, cal_summary, hold_df, CAL_END, OBS_END)

mae = np.abs(comparison['error']).mean()
bias = comparison['error'].mean()
print(f'Holdout period: {CAL_END.date()} → {OBS_END.date()}')
print(f'Holdout MAE  : {mae:.4f} purchases per customer')
print(f'Holdout Bias : {bias:+.4f} (positive = model over-predicts)')

ch_map_for_comparison = channel_map if has_channels else None

if has_channels:
    comp_ch = comparison.copy()
    comp_ch['Channel'] = comp_ch.index.map(channel_map)
    ch_mae = comp_ch.groupby('Channel')['error'].apply(lambda x: np.abs(x).mean()).round(4)
    print('\nHoldout MAE by channel:')
    print(ch_mae.to_string())

fig = plot_holdout_validation(
    comparison,
    channel_map=ch_map_for_comparison,
    save_path=OUTPUTS / 'holdout_validation.png',
)
plt.show()

print('\nValidation note: individual-level predictions are inherently noisy (Poisson process).')
print('Aggregate bias can cancel. Channel MAE does not demonstrate channel-level unbiasedness.')

## Generated channel estimates

Mean predicted revenues are GBP255.26, GBP121.12 and GBP48.60 for Enthusiast, Mainstream and Deal-Seeker. Channel differences are planted. Deal-Seeker mean P(alive) is about .89, while 92.7% exceed .5. Zero-repeat P(alive) and zero CLV are different model conventions, not evidence of loyalty or worthlessness.


In [ ]:
if has_channels:
    ch_clv = clv_by_channel(clv_df, channel_map)
    print('Nominal annual predicted revenue by generated channel (endpoint unverified):')
    print(ch_clv[['Channel', 'n_customers', 'mean_clv', 'median_clv', 'p90_clv',
                   'mean_prob_alive', 'pct_alive', 'mean_predicted_purchases']].to_string(index=False))

    fig = plot_channel_clv_boxplot(
        clv_df, channel_map,
        save_path=OUTPUTS / 'channel_clv_comparison.png',
    )
    plt.show()

    enthusiast_clv = float(ch_clv.loc[ch_clv['Channel'] == 'Enthusiast', 'mean_clv'].iloc[0])
    deal_clv = float(ch_clv.loc[ch_clv['Channel'] == 'Deal-Seeker', 'mean_clv'].iloc[0])
    ratio = enthusiast_clv / max(deal_clv, 0.01)
    print(f'\nEnthusiast mean CLV is {ratio:.1f}× higher than Deal-Seeker mean CLV.')
else:
    print('Channel data not available.')

## Hypothetical campaign heuristic

A mean-CLV, fraction-above-survival-cutoff and assumed-response product is not incremental ROI. Survival is already integrated in expected purchases. No contacted/uncontacted effect or margin model exists. The GBP500,000 setting exceeds the one-contact population, which costs GBP8,750. No actual allocation is recommended.


In [ ]:
if has_channels:
    roi_df = campaign_heuristic_by_channel(
        clv_df, channel_map, cost_per_contact=cfg['campaign']['cost_per_contact'],
        response_rates=cfg['campaign']['response_rates'],
    )
    print('Hypothetical revenue-weighting heuristic, not incremental campaign ROI:')
    print(roi_df.to_string(index=False))
    print(f"One contact per generated customer costs GBP {len(clv_df) * cfg['campaign']['cost_per_contact']:,.2f}.")
    print('No campaign allocation is justified without incremental contribution and eligibility assumptions.')


## Decision limits

No marketing investment, acquisition policy or exclusion decision is validated. Evaluate model assumptions, observed-zero retention, single-purchase treatment and future spend. Campaign optimization needs incremental contribution, eligibility, one-contact limits and budget accounting.


In [ ]:
print("Fictional retailer: decision limits")
print("Purchase-count holdout does not validate future spend or incremental campaign value.")
print("Enthusiasts rank highest on both saved mean Monetary and predicted CLV.")
print("The assumed budget is not managed spending and no full-budget allocation is implemented.")


## Export

Schema version 2 calls the campaign calculation a heuristic, documents the forced-zero policy, retention omission and unverified exact horizon. Selected summary fields do not contain every numerical result.


In [ ]:
run_summary: dict = {
    "schema_version": 2,
    "value_target": "nominal annual discounted revenue; exact executed horizon not verified",
    "retention_definition": "mean of available cohort cells; observed zeros omitted",
    "single_purchase_value_policy": "forced zero",
    'scenario': cfg['scenario']['name'],
    'n_customers': int(df['Customer ID'].nunique()),
    'n_transactions': int(len(df)),
    'n_invoices': int(df['Invoice'].nunique()),
    'observation_end': str(OBS_END.date()),
    'calibration_end': str(CAL_END.date()),
    'holdout_days': int((OBS_END - CAL_END).days),
    'clv_stats': stats,
    'holdout_mae': round(float(np.abs(comparison['error']).mean()), 4),
    'holdout_bias': round(float(comparison['error'].mean()), 4),
    'month1_retention_overall': round(float(month1_overall), 4) if not np.isnan(month1_overall) else None,
    'guardrail_threshold': guardrail,
    'guardrail_breached': bool(month1_overall < guardrail) if not np.isnan(month1_overall) else None,
    'n_segments': int(seg['Segment'].nunique()),
    'top_segment_by_revenue': str(seg.iloc[0]['Segment']),
    'bgnbd_params': {k: round(float(v), 4) for k, v in bgf.params_.items()},
}

if has_channels:
    ch_clv_final = clv_by_channel(clv_df, channel_map)
    ret_sum_final = month1_retention_summary(df, freq=cfg['cohorts']['frequency'])
    roi_final = campaign_heuristic_by_channel(
        clv_df, channel_map,
        cost_per_contact=cfg['campaign']['cost_per_contact'],
        response_rates=cfg['campaign']['response_rates'],
    )

    run_summary['channel_clv'] = {
        str(row['Channel']): {
            'n_customers': int(row['n_customers']),
            'mean_clv': float(row['mean_clv']),
            'median_clv': float(row['median_clv']),
            'p90_clv': float(row['p90_clv']),
            'total_clv': float(row['total_clv']),
            'mean_prob_alive': float(row['mean_prob_alive']),
            'pct_alive': float(row['pct_alive']),
        }
        for _, row in ch_clv_final.iterrows()
    }

    run_summary['month1_retention_by_channel'] = {
        str(row['Channel']): round(float(row['mean_month1_retention']), 4)
        for _, row in ret_sum_final.iterrows()
    }

    run_summary['campaign_heuristic_by_channel'] = {
        str(row['Channel']): {
            'heuristic_score': float(row['heuristic_score']),
            'net_per_contact': float(row['net_per_contact']),
            'revenue_proxy_per_contact': float(row['revenue_proxy_per_contact']),
        }
        for _, row in roi_final.iterrows()
    }

(OUTPUTS / 'run_summary.json').write_text(json.dumps(run_summary, indent=2))
print('Outputs written:')
for f in sorted(OUTPUTS.iterdir()):
    print(f'  {f.name}')
print()
print(json.dumps(run_summary, indent=2))